In [1]:
import pandas as pd
import numpy as np

In [2]:
upward_df = pd.read_csv("Data/upward.csv")

In [3]:
upward_df.loc[upward_df['tic'] == 'STOCK']

,date,tic,close,return_t-1,return_t-2,return_t-3,ma_3,ma_5,ma_10,log_return,volatility_5,momentum_5
991,2023-01-10,STOCK,109,0.009259,0.018692,0.028302,108.0,107.0,104.5,0.009217,0.000139,5.0
992,2023-01-11,STOCK,110,0.009174,0.018519,0.028037,109.0,108.0,105.5,0.009132,0.000137,5.0
993,2023-01-12,STOCK,111,0.009091,0.018349,0.027778,110.0,109.0,106.5,0.009050,0.000134,5.0
994,2023-01-13,STOCK,112,0.009009,0.018182,0.027523,111.0,110.0,107.5,0.008969,0.000132,5.0
995,2023-01-14,STOCK,113,0.008929,0.018018,0.027273,112.0,111.0,108.5,0.008889,0.000130,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1977,2025-09-22,STOCK,1095,0.000914,0.001830,0.002747,1094.0,1093.0,1090.5,0.000914,0.000001,5.0
1978,2025-09-23,STOCK,1096,0.000913,0.001828,0.002745,1095.0,1094.0,1091.5,0.000913,0.000001,5.0
1979,2025-09-24,STOCK,1097,0.000912,0.001826,0.002742,1096.0,1095.0,1092.5,0.000912,0.000001,5.0
1980,2025-09-25,STOCK,1098,0.000912,0.001825,0.002740,1097.0,1096.0,1093.5,0.000911,0.000001,5.0


giorno 0:  weights 50% 50% (converted to 1,0), stock value (10/01/23): 109,  portfolio value = 1000
+ step di 1
giorno 1: weights 50% 50%,(1,0) stock value (11/01/23): 110,  portfolio value = 1000
+ step di 1
giorno 2: weights (0,1), stock value (12/01/23): 111, portfolio_value = 1,009.09

ma siccome il day 1 è sempre fisso per via dell'inizializzazione forzata, devo iniziare da il day 2 fino al giorno 990 

In [4]:
C = 1000
weight = 1
R = 1099/110

v = C * (weight * R + (1-weight)* 1)
v

9990.90909090909

In [5]:
upward_run_work = pd.read_csv("Results/250619_hc_binary_udp/ddpg/upward/training_logs/01.csv")

In [6]:
optimal_upward_df_binary = upward_run_work.loc[(upward_run_work['episode'] == 50), ['day', 'allocation_weights', 'new_portfolio_value']]

In [7]:
optimal_upward_df_binary

,day,allocation_weights,new_portfolio_value
48510,1,"1.00, 0.00","1,000.00"
48511,2,"0.00, 1.00","1,009.09"
48512,3,"0.00, 1.00","1,018.18"
48513,4,"0.00, 1.00","1,027.27"
48514,5,"0.00, 1.00","1,036.36"
...,...,...,...
49495,986,"0.00, 1.00","9,954.55"
49496,987,"0.00, 1.00","9,963.64"
49497,988,"0.00, 1.00","9,972.73"
49498,989,"0.00, 1.00","9,981.82"


In [8]:
len(optimal_upward_df_binary)

990

In [9]:
C = 1000
weight = 0
R = 1/991

v = C * (weight * R + (1-weight)* 1)
v

1000.0

In [10]:
optimal_downward_df_binary = upward_run_work.loc[(upward_run_work['episode'] == 50), ['day', 'allocation_weights', 'new_portfolio_value']]
optimal_downward_df_binary['new_portfolio_value'] = 1000
optimal_downward_df_binary["allocation_weights"] = [[1, 0]] * len(optimal_downward_df_binary)
optimal_downward_df_binary

,day,allocation_weights,new_portfolio_value
48510,1,"[1, 0]",1000
48511,2,"[1, 0]",1000
48512,3,"[1, 0]",1000
48513,4,"[1, 0]",1000
48514,5,"[1, 0]",1000
...,...,...,...
49495,986,"[1, 0]",1000
49496,987,"[1, 0]",1000
49497,988,"[1, 0]",1000
49498,989,"[1, 0]",1000


In [11]:
periodic_df = pd.read_csv("Data/periodic.csv")

In [12]:
periodic_df.loc[periodic_df['tic'] =='STOCK'].iloc[:5]

,date,tic,close,return_t-1,return_t-2,return_t-3,ma_3,ma_5,ma_10,log_return,volatility_5,momentum_5
991,2023-01-10,STOCK,546.601954,0.005946,0.013791,0.023442,543.046487,538.290705,523.544972,0.005928,0.002549,24.853678
992,2023-01-11,STOCK,548.786168,0.003996,0.009966,0.017842,546.253094,542.401513,528.423588,0.003988,0.002785,20.554044
993,2023-01-12,STOCK,549.874749,0.001984,0.005988,0.011969,548.420957,545.560076,533.411063,0.001982,0.002969,15.792811
994,2023-01-13,STOCK,549.843251,-0.000057,0.001926,0.005930,549.501390,547.695457,537.648198,-0.000057,0.003099,10.676906
995,2023-01-14,STOCK,548.692382,-0.002093,-0.002150,-0.000171,549.470127,548.759701,541.039835,-0.002095,0.003177,5.321220


In [13]:
class PeriodicTrendPriceGenerator:
    def __init__(self, start=500, amplitude=50, frequency=0.15):
        self.start = start
        self.amplitude = amplitude
        self.frequency = frequency
        self.t = 0

    def generate_price(self, last_price=None):
        value = self.amplitude * np.sin(self.frequency * self.t) + self.start
        self.t += 1
        return value

# Generate 991 days of stock prices
days = 999
generator = PeriodicTrendPriceGenerator(start= 500, amplitude=50, frequency=0.15)
stock_prices = [generator.generate_price() for _ in range(days)]

stock_prices = np.array(stock_prices)[8:]
allocation = [1,0]
portfolio_value = 1000
days_df = []
optimal_allocations = []
optimal_portfolio_values = []

for day, stock_price in enumerate(stock_prices):
    print("day: ", day)
    if day == 0:
       pass
    else:
        if day == 1:
           
            assert round(stock_prices[day]) == 549
        old_allocation = allocation
        
        print("stock price today", stock_prices[day])
        print("stock price yesterday", stock_prices[day-1])
        portfolio_return = sum((np.array([500, stock_prices[day]]) / np.array([500, stock_prices[day-1]])-1)*np.array(old_allocation))
        new_portfolio_value = portfolio_value*(1+portfolio_return)
        portfolio_value = new_portfolio_value
        print("portfolio_value", portfolio_value)
        if (stock_prices[day] > stock_prices[day-1]):
            allocation = [0, 1]
        else:
            allocation = [1, 0]
        print("old_allocation", old_allocation)
        print("allocation:", allocation)
        days_df.append(day)
        optimal_allocations.append(old_allocation)
        optimal_portfolio_values.append(portfolio_value)

optimal_periodic_df_binary = pd.DataFrame({"day": days_df,
                                            "allocation_weights":optimal_allocations,
                                            "new_portfolio_value": optimal_portfolio_values
                                           })

day:  0
day:  1
stock price today 548.786167891333
stock price yesterday 546.6019542983613
portfolio_value 1000.0
old_allocation [1, 0]
allocation: [0, 1]
day:  2
stock price today 549.8747493302027
stock price yesterday 548.786167891333
portfolio_value 1001.9836167574204
old_allocation [0, 1]
allocation: [0, 1]
day:  3
stock price today 549.843251422696
stock price yesterday 549.8747493302027
portfolio_value 1001.9262211644743
old_allocation [0, 1]
allocation: [1, 0]
day:  4
stock price today 548.6923815439097
stock price yesterday 549.843251422696
portfolio_value 1001.9262211644743
old_allocation [1, 0]
allocation: [1, 0]
day:  5
stock price today 546.4479857501934
stock price yesterday 548.6923815439097
portfolio_value 1001.9262211644743
old_allocation [1, 0]
allocation: [1, 0]
day:  6
stock price today 543.1604683324437
stock price yesterday 546.4479857501934
portfolio_value 1001.9262211644743
old_allocation [1, 0]
allocation: [1, 0]
day:  7
stock price today 538.9036598443961
stoc

In [14]:
optimal_periodic_df_binary 

,day,allocation_weights,new_portfolio_value
0,1,"[1, 0]",1000.000000
1,2,"[0, 1]",1001.983617
2,3,"[0, 1]",1001.926221
3,4,"[1, 0]",1001.926221
4,5,"[1, 0]",1001.926221
...,...,...,...
985,986,"[1, 0]",95929.842356
986,987,"[1, 0]",95929.842356
987,988,"[1, 0]",95929.842356
988,989,"[0, 1]",96323.358112


In [15]:
upward_exp_4 = pd.read_csv("Data/optimal_portfolio_values_calculations/upward_exp_4.csv")

In [16]:
upward_exp_4.loc[upward_exp_4['tic'] == 'STOCK'].iloc[:991]

,date,tic,close
1000,2023-01-01,STOCK,100
1001,2023-01-02,STOCK,101
1002,2023-01-03,STOCK,102
1003,2023-01-04,STOCK,103
1004,2023-01-05,STOCK,104
...,...,...,...
1986,2025-09-13,STOCK,1086
1987,2025-09-14,STOCK,1087
1988,2025-09-15,STOCK,1088
1989,2025-09-16,STOCK,1089


In [17]:
C = 1005
weight = 1
R = 1090/101
v = C * (weight * R + (1-weight)* 1)
v

10846.039603960397

In [18]:
upward_run_work_exp_4 = pd.read_csv("Data/optimal_portfolio_values_calculations/run_worked_upward_exp_4.csv")

In [19]:
upward_run_work_exp_4.loc[(upward_run_work_exp_4['episode'] == 500)].iloc[:990]

,episode,day,actions,allocation_weights,portfolio_return,reward,new_portfolio_value
498501,500,1,"0.50, 0.50","0.50, 0.50",0.50%,1005.00,"1,005"
498502,500,2,"0.00, 1.00","0.00, 1.00",0.99%,1014.95,"1,015"
498503,500,3,"0.00, 1.00","0.00, 1.00",0.98%,1024.90,"1,025"
498504,500,4,"0.00, 1.00","0.00, 1.00",0.97%,1034.85,"1,035"
498505,500,5,"0.00, 1.00","0.00, 1.00",0.96%,1044.80,"1,045"
...,...,...,...,...,...,...,...
499486,500,986,"0.00, 1.00","0.00, 1.00",0.09%,10806.24,"10,806"
499487,500,987,"0.00, 1.00","0.00, 1.00",0.09%,10816.19,"10,816"
499488,500,988,"0.00, 1.00","0.00, 1.00",0.09%,10826.14,"10,826"
499489,500,989,"0.00, 1.00","0.00, 1.00",0.09%,10836.09,"10,836"


In [20]:
optimal_upward_df_exp_4 = upward_run_work_exp_4.loc[(upward_run_work_exp_4['episode'] == 500)].iloc[:990].loc[:, ['day', 'allocation_weights', 'new_portfolio_value']]

In [21]:
downward_exp_4 = pd.read_csv("Data/optimal_portfolio_values_calculations/downward_exp_4.csv")

In [22]:
downward = pd.read_csv("Data/downward.csv")

In [23]:
downward_exp_4.loc[downward_exp_4['tic'] == 'STOCK'].iloc[1:992] #da 999 a 9

,date,tic,close
1001,2023-01-02,STOCK,999
1002,2023-01-03,STOCK,998
1003,2023-01-04,STOCK,997
1004,2023-01-05,STOCK,996
1005,2023-01-06,STOCK,995
...,...,...,...
1987,2025-09-14,STOCK,13
1988,2025-09-15,STOCK,12
1989,2025-09-16,STOCK,11
1990,2025-09-17,STOCK,10


In [24]:
downward_run_work = pd.read_csv("Data/optimal_portfolio_values_calculations/run_worked_downward_exp_4.csv")

In [25]:
C = 1000
weight = 0
R = 999/9
v = C * (weight * R + (1-weight)* 1)
v

1000.0

In [26]:
optimal_downward_df_exp_4 = downward_run_work.loc[(downward_run_work['episode'] == 500)].iloc[:990]

In [27]:
periodic_exp_4_df = pd.read_csv("Data/optimal_portfolio_values_calculations/periodic_exp_4.csv")

In [28]:
periodic_exp_4_df.loc[periodic_exp_4_df['tic'] =='STOCK'].iloc[:5]

,date,tic,close
1000,2023-01-01,STOCK,500.000000
1001,2023-01-02,STOCK,500.000000
1002,2023-01-03,STOCK,507.471907
1003,2023-01-04,STOCK,514.776010
1004,2023-01-05,STOCK,521.748277


In [29]:

stock_prices

array([546.6019543 , 548.78616789, 549.87474933, 549.84325142,
       548.69238154, 546.44798575, 543.16046833, 538.90365984,
       533.77315903, 527.88418587, 521.36899401, 514.37390062,
       507.0560004 , 499.57963763, 492.11271529, 484.82292436,
       477.87397784, 471.42193406, 465.61169204, 460.57373728,
       456.42121138, 453.24737112, 451.12349412, 450.09727806,
       450.19176956, 451.40484653, 453.70926588, 457.05327533,
       461.36177562, 466.53800714, 472.46572287, 479.01179911,
       486.02922509, 493.36040456, 500.84069502, 508.30210529,
       515.57706818, 522.50220369, 528.92198822, 534.69224725,
       539.68339319, 543.78333568, 546.89999884, 548.96338913,
       549.92716727, 549.76968886, 548.49449054, 546.13021051,
       542.7299454 , 538.37005784, 533.1484615 , 527.18242218,
       520.60592426, 513.56666171, 506.22272118, 498.73903174,
       491.28366094, 484.02404032, 477.12320531, 470.73613379,
       465.00626562, 460.06228128, 456.015212  , 452.95

In [30]:
class PeriodicTrendPriceGenerator:
    def __init__(self, start=500, amplitude=50, frequency=0.15):
        self.start = start
        self.amplitude = amplitude
        self.frequency = frequency
        self.t = 0

    def generate_price(self, last_price=None):
        value = self.amplitude * np.sin(self.frequency * self.t) + self.start
        self.t += 1
        return value

# Generate 991 days of stock prices
days = 990
generator = PeriodicTrendPriceGenerator(start= 500, amplitude=50, frequency=0.15)
stock_prices = [generator.generate_price() for _ in range(days)]
stock_prices = [500] + stock_prices
stock_prices = np.array(stock_prices)
allocation = [0.5,0.5]
portfolio_value = 1000
days_df = []
optimal_allocations = []
optimal_portfolio_values = []

for day, stock_price in enumerate(stock_prices):
    print("day: ", day)
    if day == 0:
       pass
    else:
        if day == 1:
           
            pass
        old_allocation = allocation
        
        print("stock price today", stock_prices[day])
        print("stock price yesterday", stock_prices[day-1])
        portfolio_return = sum((np.array([500, stock_prices[day]]) / np.array([500, stock_prices[day-1]])-1)*np.array(old_allocation))
        new_portfolio_value = portfolio_value*(1+portfolio_return)
        portfolio_value = new_portfolio_value
        print("portfolio_value", portfolio_value)
        if (stock_prices[day] >= stock_prices[day-1]):
            allocation = [0, 1]
        else:
            allocation = [1, 0]
        print("old_allocation", old_allocation)
        print("allocation:", allocation)
        days_df.append(day)
        optimal_allocations.append(old_allocation)
        optimal_portfolio_values.append(portfolio_value)

optimal_periodic_df_exp_4 = pd.DataFrame({"day": days_df,
                                            "optimal allocation":optimal_allocations,
                                            "optimal portfolio value": optimal_portfolio_values
                                           })


day:  0
day:  1
stock price today 500.0
stock price yesterday 500.0
portfolio_value 1000.0
old_allocation [0.5, 0.5]
allocation: [0, 1]
day:  2
stock price today 507.47190662367996
stock price yesterday 500.0
portfolio_value 1014.9438132473598
old_allocation [0, 1]
allocation: [0, 1]
day:  3
stock price today 514.776010333067
stock price yesterday 507.47190662367996
portfolio_value 1029.552020666134
old_allocation [0, 1]
allocation: [0, 1]
day:  4
stock price today 521.7482767055615
stock price yesterday 514.776010333067
portfolio_value 1043.496553411123
old_allocation [0, 1]
allocation: [0, 1]
day:  5
stock price today 528.2321236697518
stock price yesterday 521.7482767055615
portfolio_value 1056.4642473395036
old_allocation [0, 1]
allocation: [0, 1]
day:  6
stock price today 534.0819380011667
stock price yesterday 528.2321236697518
portfolio_value 1068.1638760023336
old_allocation [0, 1]
allocation: [0, 1]
day:  7
stock price today 539.1663454813742
stock price yesterday 534.08193800

In [31]:
upward_run_work

,episode,day,actions,allocation_weights,portfolio_return,reward,new_portfolio_value
0,1,1,"0.50, 0.50","1.00, 0.00",0.92%,-0.009174,"1,000.00"
1,1,2,"0.61, 0.97","0.00, 1.00",0.91%,0.009091,"1,009.09"
2,1,3,"0.87, 0.81","1.00, 0.00",0.90%,-0.009009,"1,009.09"
3,1,4,"0.29, 0.51","0.00, 1.00",0.89%,0.008929,"1,018.10"
4,1,5,"0.15, 0.89","0.00, 1.00",0.88%,0.008850,"1,027.11"
...,...,...,...,...,...,...,...
49945,51,446,"0.00, 1.00","0.00, 1.00",0.18%,0.001805,"5,045.45"
49946,51,447,"0.00, 1.00","0.00, 1.00",0.18%,0.001802,"5,054.55"
49947,51,448,"0.00, 1.00","0.00, 1.00",0.18%,0.001799,"5,063.64"
49948,51,449,"0.00, 1.00","0.00, 1.00",0.18%,0.001795,"5,072.73"


In [32]:
upward_run_work['allocation_weights']

0        1.00, 0.00
1        0.00, 1.00
2        1.00, 0.00
3        0.00, 1.00
4        0.00, 1.00
            ...    
49945    0.00, 1.00
49946    0.00, 1.00
49947    0.00, 1.00
49948    0.00, 1.00
49949    0.00, 1.00
Name: allocation_weights, Length: 49950, dtype: object

In [33]:
upward_run_work

,episode,day,actions,allocation_weights,portfolio_return,reward,new_portfolio_value
0,1,1,"0.50, 0.50","1.00, 0.00",0.92%,-0.009174,"1,000.00"
1,1,2,"0.61, 0.97","0.00, 1.00",0.91%,0.009091,"1,009.09"
2,1,3,"0.87, 0.81","1.00, 0.00",0.90%,-0.009009,"1,009.09"
3,1,4,"0.29, 0.51","0.00, 1.00",0.89%,0.008929,"1,018.10"
4,1,5,"0.15, 0.89","0.00, 1.00",0.88%,0.008850,"1,027.11"
...,...,...,...,...,...,...,...
49945,51,446,"0.00, 1.00","0.00, 1.00",0.18%,0.001805,"5,045.45"
49946,51,447,"0.00, 1.00","0.00, 1.00",0.18%,0.001802,"5,054.55"
49947,51,448,"0.00, 1.00","0.00, 1.00",0.18%,0.001799,"5,063.64"
49948,51,449,"0.00, 1.00","0.00, 1.00",0.18%,0.001795,"5,072.73"


In [34]:
optimal_periodic_df_exp_4

,day,optimal allocation,optimal portfolio value
0,1,"[0.5, 0.5]",1000.000000
1,2,"[0, 1]",1014.943813
2,3,"[0, 1]",1029.552021
3,4,"[0, 1]",1043.496553
4,5,"[0, 1]",1056.464247
...,...,...,...
985,986,"[1, 0]",105289.941146
986,987,"[1, 0]",105289.941146
987,988,"[1, 0]",105289.941146
988,989,"[1, 0]",105289.941146


In [35]:
def clean_allocation_df(df):
    allocation_column = df.filter(regex = 'allocation').columns[0]
    portfolio_column = df.filter(regex = 'portfolio').columns[0]
    # Convert allocation_weights to [1, 0] or [0, 1] format
    if df[allocation_column].dtype == object:
        df[allocation_column] = df[allocation_column].apply(
        lambda x: [int(i) for i in x] if isinstance(x, (list, tuple)) else [int(float(i.strip())) for i in str(x).split(",")]
    )

    # Clean and convert new_portfolio_value to float with 2 decimals
    df[portfolio_column] = (
        df[portfolio_column]
        .astype(str)                      # ensure string for cleanup
        .str.replace(",", "", regex=False)  # remove thousand separators
        .astype(float)
        .round(2)
    )

    return df

optimal_upward_df_binary = clean_allocation_df(optimal_upward_df_binary)
optimal_periodic_df_binary = clean_allocation_df(optimal_periodic_df_binary)
optimal_downward_df_binary = clean_allocation_df(optimal_downward_df_binary)

In [36]:
optimal_upward_df_binary.head()

,day,allocation_weights,new_portfolio_value
48510,1,"[1, 0]",1000.00
48511,2,"[0, 1]",1009.09
48512,3,"[0, 1]",1018.18
48513,4,"[0, 1]",1027.27
48514,5,"[0, 1]",1036.36


In [37]:
import pandas as pd
import numpy as np
import glob
import os

# Define or load this beforehand
# optimal_upward_df_binary = pd.read_csv("path_to/optimal_upward_df_binary.csv")
# optimal_upward_df_binary = clean_allocation_df(optimal_upward_df_binary)

base_path = "Results/250619_hc_binary_udp"
agents = ["a2c", "ddpg", "ppo"]
regime = "upward"

# Placeholder dictionaries
result_dfs = {}
all_data = {}

# Function to compute mean L2 error
def compute_mean_l2_error(agent_allocs, opt_allocs):
    errors = np.linalg.norm(agent_allocs - opt_allocs, axis=1)
    return np.mean(errors)

# Clean the optimal dataframe before loop
optimal_upward_df_binary = clean_allocation_df(optimal_upward_df_binary)
opt_allocs_array = np.array(optimal_upward_df_binary["allocation_weights"].tolist())

for agent in agents:
    csv_pattern = os.path.join(base_path, agent, regime, "training_logs", "*.csv")
    csv_files = sorted(glob.glob(csv_pattern))
    
    dfs = []
    mean_errors = []  # Re-initialize for each agent

    for i, fpath in enumerate(csv_files, start=1):
        df = pd.read_csv(fpath)
        df = df.loc[df['episode'] == 50, ['day', 'allocation_weights', 'new_portfolio_value']]
        df = clean_allocation_df(df)

        agent_allocs_array = np.array(df["allocation_weights"].tolist())
        print(agent_allocs_array)
        print(opt_allocs_array)
        # Align both arrays by length (truncate to minimum length)
        min_len = min(len(agent_allocs_array), len(opt_allocs_array))
        print(len(agent_allocs_array))
        print(len(opt_allocs_array))
        error = compute_mean_l2_error(agent_allocs_array[:min_len], opt_allocs_array[:min_len])
        mean_errors.append(error)
        dfs.append(df)

    result_dfs[agent] = pd.DataFrame({
        "run": list(range(1, len(mean_errors) + 1)),
        "mean_allocation_error": mean_errors
    })

    all_data[agent] = pd.concat(dfs, ignore_index=True)


[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [1 0]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [1 0]
 ...
 [1 0]
 [0 1]
 [1 0]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
990
990
[[1 0]
 [0 1]
 [0 1]
 ...
 [0 1]
 [0 1]
 [0 1]]
[[1 0]
 [0 1]
 [

In [38]:
optimal_periodic_df_binary

,day,allocation_weights,new_portfolio_value
0,1,"[1, 0]",1000.00
1,2,"[0, 1]",1001.98
2,3,"[0, 1]",1001.93
3,4,"[1, 0]",1001.93
4,5,"[1, 0]",1001.93
...,...,...,...
985,986,"[1, 0]",95929.84
986,987,"[1, 0]",95929.84
987,988,"[1, 0]",95929.84
988,989,"[0, 1]",96323.36


In [39]:
import pandas as pd
import numpy as np
import glob
import os

base_path = "Results/250619_hc_binary_udp"
base_path_noisy = "Results/250619_hc_binary_udp_noise"
agents = ["a2c", "ddpg", "ppo"]
regimes = ["upward", "downward", "periodic", "upward_noise", "downward_noise", "periodic_noise"]

# Result containers
result_dfs = {}  # structure: result_dfs[agent][regime]
all_data = {}

# Utility function to clean 'allocation_weights'
def clean_allocation_df(df):
    df = df.copy()
    df["allocation_weights"] = df["allocation_weights"].apply(
        lambda x: [float(i) for i in str(x).strip("[]").split(",")]
    )
    return df

# Function to compute mean L2 error
def compute_mean_l2_error(agent_allocs, opt_allocs):
    errors = np.linalg.norm(agent_allocs - opt_allocs, axis=1)
    return np.mean(errors)

# You must define or load the optimal allocation DataFrames for each regime
optimal_data = {
    "upward": clean_allocation_df(optimal_upward_df_binary),
    "downward": clean_allocation_df(optimal_downward_df_binary),
    "periodic": clean_allocation_df(optimal_periodic_df_binary),
     "upward_noise": clean_allocation_df(optimal_upward_df_binary),
    "downward_noise": clean_allocation_df(optimal_downward_df_binary),
    "periodic_noise": clean_allocation_df(optimal_periodic_df_binary),
}

for agent in agents:
    result_dfs[agent] = {}
    all_data[agent] = {}
    
    for regime in regimes:
        # Prepare optimal allocations
        opt_allocs_array = np.array(optimal_data[regime]["allocation_weights"].tolist())

        # Gather CSVs
        # Select correct base path based on regime type
        current_base = base_path_noisy if "noise" in regime else base_path
        csv_pattern = os.path.join(current_base, agent, regime, "training_logs", "*.csv")

        csv_files = sorted(glob.glob(csv_pattern))
        print(csv_files)
        mean_errors = []
        dfs = []
        delta_vs = []

        for fpath in csv_files:
            df = pd.read_csv(fpath)
            df = df.loc[df['episode'] == 50, ['day', 'allocation_weights', 'new_portfolio_value']]
            df = clean_allocation_df(df)

            agent_allocs_array = np.array(df["allocation_weights"].tolist())
            min_len = min(len(agent_allocs_array), len(opt_allocs_array))
            opt_df = optimal_data[regime]

            error = compute_mean_l2_error(agent_allocs_array[:min_len], opt_allocs_array[:min_len])
            mean_errors.append(error)
            dfs.append(df)
              # Compute ∆V
            try:
                V_agent_T = float(str(df["new_portfolio_value"].iloc[-1]).replace(",", ""))
                V_opt_T = float(str(opt_df["new_portfolio_value"].iloc[min_len - 1]).replace(",", ""))
                delta_v = V_agent_T - V_opt_T
            except Exception as e:
                delta_v = np.nan  # fallback if any issue
            delta_vs.append(delta_v)

        result_dfs[agent][regime] = pd.DataFrame({
            "run": list(range(1, len(mean_errors) + 1)),
            "mean_allocation_error": mean_errors,
            "delta_V": delta_vs
        })
        all_data[agent][regime] = pd.concat(dfs, ignore_index=True)

# Example: access PPO's results on periodic regime
print(result_dfs["ppo"]["periodic"])


['Results/250619_hc_binary_udp/a2c/upward/training_logs/01.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/02.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/03.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/04.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/05.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/06.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/07.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/08.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/09.csv', 'Results/250619_hc_binary_udp/a2c/upward/training_logs/10.csv']
['Results/250619_hc_binary_udp/a2c/downward/training_logs/01.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/02.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/03.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/04.csv', 'Results/250619_hc_binary_udp/a2c/downward/training_logs/05.csv', 'Results/250619_hc_binary_ud

In [46]:
result_dfs['ppo']['downward_noise']

,run,mean_allocation_error,delta_V
0,1,0.451406,-408.83
1,2,0.564257,3.64
2,3,0.631396,128.05
3,4,0.282843,310.00
4,5,0.368553,-146.14
5,6,0.435692,42.19
6,7,0.269986,107.98
7,8,0.324269,34.04
8,9,0.511402,-105.38
9,10,0.402837,193.06


In [49]:
for agent in ["a2c", "ddpg", "ppo"]:
    print("\\begin{table}[ht]")
    print("  \\centering")
    print(f"  \\caption{{Expt. 3 - {agent.upper()} averaged results across 10 runs with different generators}}")
    print(f"  \\label{{tab:{agent}_results}}")

    # --- Clean Regimes Subtable ---
    print("  \\begin{subtable}[t]{0.48\\textwidth}")
    print("  \\centering")
    print("  \\caption{Generators with no Noise}")
    print("  \\begin{tabular}{ccc}")
    print("  \\toprule")
    print("  Regime & Mean Allocation Error & $\\Delta V$ \\\\")
    print("  \\midrule")
    for regime in ["upward", "downward", "periodic"]:
        df = result_dfs[agent].get(regime)
        if df is not None and not df.empty:
            mean_error = df["mean_allocation_error"].mean()
            delta_v = df["delta_V"].mean()
            print(f"  {regime.capitalize()} & {mean_error:.2f} & {delta_v:.2f} \\\\")
    print("  \\bottomrule")
    print("  \\end{tabular}")
    print("  \\end{subtable}")

    print("  \\hfill")

    # --- Noisy Regimes Subtable ---
    print("  \\begin{subtable}[t]{0.48\\textwidth}")
    print("  \\centering")
    print("  \\caption{Generators with Noise}")
    print("  \\begin{tabular}{ccc}")
    print("  \\toprule")
    print("  Regime & Mean Allocation Error & $\\Delta V$ \\\\")
    print("  \\midrule")
    for regime in ["upward_noise", "downward_noise", "periodic_noise"]:
        df = result_dfs[agent].get(regime)
        if df is not None and not df.empty:
            mean_error = df["mean_allocation_error"].mean()
            delta_v = df["delta_V"].mean() #np.mean(np.abs(df["delta_V"]))
            print(f"  {regime.replace('_noise','').capitalize()} & {mean_error:.2f} & {delta_v:.2f} \\\\")
    print("  \\bottomrule")
    print("  \\end{tabular}")
    print("  \\end{subtable}")

    print("\\end{table}")
    print("\n\n")

\begin{table}[ht]
  \centering
  \caption{Expt. 3 - A2C averaged results across 10 runs with different generators}
  \label{tab:a2c_results}
  \begin{subtable}[t]{0.48\textwidth}
  \centering
  \caption{Generators with no Noise}
  \begin{tabular}{ccc}
  \toprule
  Regime & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward & 0.06 & -924.72 \\
  Downward & 0.01 & -70.81 \\
  Periodic & 0.47 & -75042.38 \\
  \bottomrule
  \end{tabular}
  \end{subtable}
  \hfill
  \begin{subtable}[t]{0.48\textwidth}
  \centering
  \caption{Generators with Noise}
  \begin{tabular}{ccc}
  \toprule
  Regime & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward & 0.09 & -6667.27 \\
  Downward & 0.20 & -92.60 \\
  Periodic & 0.72 & -95129.39 \\
  \bottomrule
  \end{tabular}
  \end{subtable}
\end{table}



\begin{table}[ht]
  \centering
  \caption{Expt. 3 - DDPG averaged results across 10 runs with different generators}
  \label{tab:ddpg_results}
  \begin{subtable}[t]{0.48\textwidth}
  \centering
 

In [50]:
# Best experiments table only
# Upward no noise -> ppo
# downward no noise -> a2c
# periodic no noise -> A2C

# upward noise -> a2c
# downward noise -> ppo
# periodic noise -> ppo 

# Define which agent is best for each regime
best_agent_per_regime = {
    "upward": "ppo",
    "downward": "a2c",
    "periodic": "a2c",
    "upward_noise": "a2c",
    "downward_noise": "ppo",
    "periodic_noise": "ppo"
}

# Define the order and labels for display
regimes_order = [
    ("upward", "Upward"),
    ("downward", "Downward"),
    ("periodic", "Periodic"),
    ("upward_noise", "Upward (Noise)"),
    ("downward_noise", "Downward (Noise)"),
    ("periodic_noise", "Periodic (Noise)")
]

# Start LaTeX table
print("\\begin{table}[ht]")
print("  \\centering")
print("  \\caption{Expt. 3 -- Best Agent per Generator Regime}")
print("  \\label{tab:best_agent_results}")
print("  \\begin{tabular}{lccc}")
print("  \\toprule")
print("  Generator & Best Agent & Mean Allocation Error & $\\Delta V$ \\\\")
print("  \\midrule")

# Fill in table rows
for regime_key, label in regimes_order:
    best_agent = best_agent_per_regime[regime_key]
    df = result_dfs[best_agent].get(regime_key)
    if df is not None and not df.empty:
        mean_error = df["mean_allocation_error"].mean()
        delta_v = df["delta_V"].mean()
        print(f"  {label} & {best_agent.upper()} & {mean_error:.2f} & {delta_v:.2f} \\\\")

print("  \\bottomrule")
print("  \\end{tabular}")
print("\\end{table}")


\begin{table}[ht]
  \centering
  \caption{Expt. 3 -- Best Agent per Generator Regime}
  \label{tab:best_agent_results}
  \begin{tabular}{lccc}
  \toprule
  Generator & Best Agent & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward & PPO & 0.04 & -572.12 \\
  Downward & A2C & 0.01 & -70.81 \\
  Periodic & A2C & 0.47 & -75042.38 \\
  Upward (Noise) & A2C & 0.09 & -6667.27 \\
  Downward (Noise) & PPO & 0.42 & 15.86 \\
  Periodic (Noise) & PPO & 0.65 & -92974.54 \\
  \bottomrule
  \end{tabular}
\end{table}
